In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from utils_ising import visualize_ising, ising_2pt_corr_direction, ising2d_ham, ising2d_swendsen_wang
from model import get_rope_vit_model, ExponentialMovingAverage
from utils_train import rnd
from utils import ess
from tqdm import tqdm

TABLEAU_COLORS = {
    'blue': '#1f77b4',
    'orange': '#ff7f0e',
    'green': '#2ca02c',
    'red': '#d62728',
    'purple': '#9467bd',
    'brown': '#8c564b',
    'pink': '#e377c2',
    'gray': '#7f7f7f',
    'olive': '#bcbd22',
    'cyan': '#17becf'
}

## Model Loading

In [ ]:
L = 16
D = L**2
device = 'cuda:0'
J = 1
h = 0
inv_temp = 'high'
beta = {'high': 0.28, 'crit': 0.4407, 'low': 0.6}[inv_temp]
ckpt_dir = f"checkpoints/L_{L}_ising/ising_{inv_temp}.pth"

model = get_rope_vit_model(L, embed_dim=64, depth=6, num_heads=4, vocab_size=3, device=device)
ema = ExponentialMovingAverage(model.parameters(), decay=0.9999)
print('Model: num of params: {}, size: {:.2f} MB'.format(
    sum(p.numel() for p in model.parameters()),
    sum(p.numel() * p.element_size() for p in model.parameters()) / (1024 ** 2)))

checkpoint = torch.load(ckpt_dir, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
ema.load_state_dict(checkpoint['ema_state_dict'])
ema.store(model.parameters())
ema.copy_to(model.parameters())
model.eval();

## Generate Samples and Compute ESS

In [ ]:
log_rnd_list = []
samples_list = []
reward = lambda S: -beta * ising2d_ham(2*S-1, J=J, h=h)

## Generate in total 4096 samples
with torch.no_grad():
    for i in tqdm(range(4)):
        samples, log_rnd = rnd(model, reward, batch_size=1024, device=device)
        log_rnd_list.append(log_rnd)
        samples_list.append(samples)
        
log_rnd_list = torch.cat(log_rnd_list, dim=0)
samples_list = torch.cat(samples_list, dim=0)
print(f"Effective Sample Size = {ess(log_rnd_list):.4f}")

In [ ]:
# Ground Truth samples from Swendsen-Wang algorithm
swen_samples = ising2d_swendsen_wang(L=L, J=1, beta=beta, B=128, num_collect=32, burn_in=2 ** 10, collect_every=128, init=None)

## Visualization of Generated Samples

In [ ]:
fig1 = visualize_ising(samples_list[:25], 5, 5)

In [ ]:
fig2 = visualize_ising(swen_samples[:25], 5, 5)

## Visualization of 2pt Correlation

In [ ]:
mdns_corr_x = np.array([ising_2pt_corr_direction(samples_list*2-1, r_x=r, r_y=0, use_x=True, use_y=False).cpu().numpy() for r in range(-L//2, L//2)])
swen_corr_x = np.array([ising_2pt_corr_direction(swen_samples, r_x=r, r_y=0, use_x=True, use_y=False) for r in range(-L//2, L//2)])

print(mdns_corr_x)
print(swen_corr_x)

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(4, 4))

ax.plot(range(-L//2,L//2), mdns_corr_x, color=TABLEAU_COLORS['red'], linestyle='--', marker=">", label='MDNS')
ax.plot(range(-L//2,L//2), swen_corr_x, color=TABLEAU_COLORS['blue'], linestyle='--', marker="<", label='SW')

ax.set_xticks(np.arange(-L//2,L//2, 2))
ax.set_xlabel(r'Distance $r$', fontsize=14)
ax.set_ylabel(r'2-point Correlation', fontsize=14)
ax.set_title("Two-point Correlation of Ising Model", fontsize=14)
ax.legend(fontsize=12)

plt.subplots_adjust(wspace=0.1, hspace=0.1, bottom=0.2)
plt.tight_layout()
plt.show()